In [ ]:
import sys
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("CSVLoader").getOrCreate()


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

pyspark_file_path = '/content/drive/MyDrive/Data/Esi/BigData/ngram.csv'

schema = StructType([
    StructField("ngram", StringType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Count", IntegerType(), True),
    StructField("Pages", IntegerType(), True),
    StructField("Books", IntegerType(), True)
])

pyspark_df = spark.read.csv(pyspark_file_path, sep='\t', header=False, schema=schema)

pyspark_df.show(5)
pyspark_df.printSchema()

+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1985|    1|    1|    1|
|! $17.95|1987|    1|    1|    1|
|! $17.95|1990|    1|    1|    1|
|! $17.95|1991|    1|    1|    1|
|! $17.95|1992|    5|    5|    5|
+--------+----+-----+-----+-----+
only showing top 5 rows
root
 |-- ngram: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Count: integer (nullable = true)
 |-- Pages: integer (nullable = true)
 |-- Books: integer (nullable = true)



In [ ]:
pyspark_df.createOrReplaceTempView("ngram_table")

### 1. Return all bigrams where the Count is greater than five

In [ ]:
sql_query = spark.sql("SELECT * FROM ngram_table WHERE Count > 5")
sql_query.show()


+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1997|    6|    5|    5|
|! $17.95|1999|   11|   10|   10|
|! $17.95|2000|   11|    9|    9|
|! $17.95|2004|   14|   14|   14|
|! $17.95|2005|   13|   13|   13|
|    ! 09|1899|    6|    6|    5|
|    ! 09|1916|    7|    7|    4|
|    ! 09|1936|    6|    6|    6|
|    ! 09|1997|    6|    5|    5|
|    ! 09|1999|   11|   10|   10|
|    ! 09|2000|   11|    9|    9|
|    ! 09|2004|   14|   14|   14|
|    ! 09|2005|   13|   13|   13|
+--------+----+-----+-----+-----+



In [ ]:
api_query = pyspark_df.filter(pyspark_df.Count > 5)
api_query.show()


+--------+----+-----+-----+-----+
|   ngram|Year|Count|Pages|Books|
+--------+----+-----+-----+-----+
|! $17.95|1997|    6|    5|    5|
|! $17.95|1999|   11|   10|   10|
|! $17.95|2000|   11|    9|    9|
|! $17.95|2004|   14|   14|   14|
|! $17.95|2005|   13|   13|   13|
|    ! 09|1899|    6|    6|    5|
|    ! 09|1916|    7|    7|    4|
|    ! 09|1936|    6|    6|    6|
|    ! 09|1997|    6|    5|    5|
|    ! 09|1999|   11|   10|   10|
|    ! 09|2000|   11|    9|    9|
|    ! 09|2004|   14|   14|   14|
|    ! 09|2005|   13|   13|   13|
+--------+----+-----+-----+-----+



### 2. Return the total number of bigrams for each year.

In [ ]:
sql_query = spark.sql("SELECT Year, SUM(Count) AS Total_Count FROM ngram_table GROUP BY Year")
sql_query.show()

+----+-----------+
|Year|Total_Count|
+----+-----------+
|1829|          3|
|1990|          2|
|1903|          1|
|1884|          5|
|1888|          2|
|1924|          1|
|2003|          4|
|1823|          1|
|2007|          4|
|1869|          1|
|1892|          2|
|1889|          2|
|1927|          1|
|1866|          1|
|1877|          2|
|2006|         10|
|1908|          2|
|1925|          2|
|1824|          1|
|1848|          1|
+----+-----------+
only showing top 20 rows


In [ ]:

result_df = pyspark_df.groupBy("Year").agg(sum("Count").alias("Total_Count"))
result_df.show()

+----+-----------+
|Year|Total_Count|
+----+-----------+
|1829|          3|
|1990|          2|
|1903|          1|
|1884|          5|
|1888|          2|
|1924|          1|
|2003|          4|
|1823|          1|
|2007|          4|
|1869|          1|
|1892|          2|
|1889|          2|
|1927|          1|
|1866|          1|
|1877|          2|
|2006|         10|
|1908|          2|
|1925|          2|
|1824|          1|
|1848|          1|
+----+-----------+
only showing top 20 rows


 ### 3. Return the bigrams with the highest Count for each year

In [ ]:
sql_query = spark.sql("SELECT Year, MAX(Count) AS highest_Count FROM ngram_table GROUP BY Year")
sql_query.show()

+----+-------------+
|Year|highest_Count|
+----+-------------+
|1829|            3|
|1990|            1|
|1903|            1|
|1884|            5|
|1888|            2|
|1924|            1|
|2003|            2|
|1823|            1|
|2007|            2|
|1869|            1|
|1892|            2|
|1889|            2|
|1927|            1|
|1866|            1|
|1877|            2|
|2006|            5|
|1908|            2|
|1925|            2|
|1824|            1|
|1848|            1|
+----+-------------+
only showing top 20 rows


In [ ]:

result_df = pyspark_df.groupBy("Year").agg(max("Count").alias("highest_Count"))
result_df.show()

+----+-------------+
|Year|highest_Count|
+----+-------------+
|1829|            3|
|1990|            1|
|1903|            1|
|1884|            5|
|1888|            2|
|1924|            1|
|2003|            2|
|1823|            1|
|2007|            2|
|1869|            1|
|1892|            2|
|1889|            2|
|1927|            1|
|1866|            1|
|1877|            2|
|2006|            5|
|1908|            2|
|1925|            2|
|1824|            1|
|1848|            1|
+----+-------------+
only showing top 20 rows


###4. Return all bigrams that appeared in 20 different years.

In [ ]:
sql_query = spark.sql(""" SELECT ngram, COUNT(DISTINCT Year) AS Year_Count  FROM ngram_table
    GROUP BY ngram  HAVING Year_Count = 20""")

sql_query.show()

+--------+----------+
|   ngram|Year_Count|
+--------+----------+
|! $17.95|        20|
+--------+----------+



In [ ]:
result = pyspark_df.groupBy("Ngram") \
    .agg(count("Year").alias("Year_Count")) \
    .filter("Year_Count == 20").distinct()

result.show()

+--------+----------+
|   Ngram|Year_Count|
+--------+----------+
|! $17.95|        20|
+--------+----------+



### 5. Return all bigrams that contain the character '!' in the first part and the character '9' in the second part (the two parts are separated by a space)

In [ ]:
sql_query = spark.sql(""" SELECT DISTINCT ngram FROM ngram_table
    WHERE ngram LIKE '!% %9%' """)

sql_query.show()

+--------+
|   ngram|
+--------+
|    ! 09|
|! $17.95|
+--------+



In [ ]:
filtered_df = pyspark_df.filtered_df = pyspark_df.filter("ngram LIKE '!% %9%'").select("ngram").distinct().show()

+--------+
|   ngram|
+--------+
|    ! 09|
|! $17.95|
+--------+



###6. Return the bigrams that appeared in all the years present in the dataset

In [ ]:
spark.sql("""
    SELECT Ngram , COUNT(DISTINCT Year) AS Year_Count
    FROM ngram_table
    GROUP BY Ngram
    HAVING Year_Count = (SELECT COUNT(DISTINCT Year) FROM ngram_table)
""").show()

+-----+----------+
|Ngram|Year_Count|
+-----+----------+
| ! 09|       100|
+-----+----------+



In [ ]:
total_years_count = pyspark_df.select("Year").distinct().count()

result_df = pyspark_df.groupBy("Ngram") \
    .agg(count("Year").alias("years_count")).distinct() \
    .filter(f"years_count == {total_years_count}").show()

+-----+-----------+
|Ngram|years_count|
+-----+-----------+
| ! 09|        100|
+-----+-----------+



### 7. Return the total number of pages and books in which each bigram appeared each available year, sorted in alphabetical order

In [ ]:
spark.sql("""
SELECT Ngram, Year, SUM(Pages) AS Total_Pages, SUM(Books) AS Total_Books
FROM ngram_table
GROUP BY Ngram, Year
ORDER BY Ngram ASC
""").show()

+--------+----+-----------+-----------+
|   Ngram|Year|Total_Pages|Total_Books|
+--------+----+-----------+-----------+
|! $17.95|2002|          5|          5|
|! $17.95|2007|          2|          2|
|! $17.95|1985|          1|          1|
|! $17.95|1990|          1|          1|
|! $17.95|1993|          2|          2|
|! $17.95|2003|          2|          2|
|! $17.95|2000|          9|          9|
|! $17.95|2005|         13|         13|
|! $17.95|1999|         10|         10|
|! $17.95|1992|          5|          5|
|! $17.95|1996|          2|          2|
|! $17.95|1997|          5|          5|
|! $17.95|1995|          1|          1|
|! $17.95|1987|          1|          1|
|! $17.95|2004|         14|         14|
|! $17.95|2006|          5|          5|
|! $17.95|1998|          3|          3|
|! $17.95|1991|          1|          1|
|! $17.95|2008|          2|          2|
|! $17.95|2001|          4|          4|
+--------+----+-----------+-----------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import sum, col


result_df = pyspark_df.groupBy("Ngram", "Year") \
    .agg(
        sum("Pages").alias("Total_Pages"),
        sum("Books").alias("Total_Books")
    ) \
    .orderBy(col("Ngram").asc())

result_df.show()

+--------+----+-----------+-----------+
|   Ngram|Year|Total_Pages|Total_Books|
+--------+----+-----------+-----------+
|! $17.95|2002|          5|          5|
|! $17.95|2007|          2|          2|
|! $17.95|1985|          1|          1|
|! $17.95|1990|          1|          1|
|! $17.95|1993|          2|          2|
|! $17.95|2003|          2|          2|
|! $17.95|2000|          9|          9|
|! $17.95|2005|         13|         13|
|! $17.95|1999|         10|         10|
|! $17.95|1992|          5|          5|
|! $17.95|1996|          2|          2|
|! $17.95|1997|          5|          5|
|! $17.95|1995|          1|          1|
|! $17.95|1987|          1|          1|
|! $17.95|2004|         14|         14|
|! $17.95|2006|          5|          5|
|! $17.95|1998|          3|          3|
|! $17.95|1991|          1|          1|
|! $17.95|2008|          2|          2|
|! $17.95|2001|          4|          4|
+--------+----+-----------+-----------+
only showing top 20 rows


### 8.Return the total number of distinct bigrams for each year, sorted indescending *order* of the year

In [ ]:
from pyspark.sql.functions import countDistinct, col

# 1. Group by Year
# 2. Count distinct values in the Ngram column
# 3. Sort by Year in descending ordera2
result_df = pyspark_df.groupBy("Year") \
    .agg(countDistinct("Ngram").alias("Distinct_Bigram_Count")) \
    .orderBy(col("Year").desc())

result_df.show()

+----+---------------------+
|Year|Distinct_Bigram_Count|
+----+---------------------+
|2008|                    2|
|2007|                    2|
|2006|                    2|
|2005|                    2|
|2004|                    2|
|2003|                    2|
|2002|                    2|
|2001|                    2|
|2000|                    2|
|1999|                    2|
|1998|                    2|
|1997|                    2|
|1996|                    2|
|1995|                    2|
|1993|                    2|
|1992|                    2|
|1991|                    2|
|1990|                    2|
|1987|                    2|
|1985|                    2|
+----+---------------------+
only showing top 20 rows


In [ ]:
spark.sql("""
    SELECT Year, COUNT(DISTINCT Ngram) AS Distinct_Bigram_Count
    FROM ngram_table
    GROUP BY Year
    ORDER BY Year DESC
""").show()

+----+---------------------+
|Year|Distinct_Bigram_Count|
+----+---------------------+
|2008|                    2|
|2007|                    2|
|2006|                    2|
|2005|                    2|
|2004|                    2|
|2003|                    2|
|2002|                    2|
|2001|                    2|
|2000|                    2|
|1999|                    2|
|1998|                    2|
|1997|                    2|
|1996|                    2|
|1995|                    2|
|1993|                    2|
|1992|                    2|
|1991|                    2|
|1990|                    2|
|1987|                    2|
|1985|                    2|
+----+---------------------+
only showing top 20 rows
